# TenaciousBench v0.1 — End-to-End Notebook

**Author:** Eyor Getachew | **Date:** 2026-05-02  
**Path:** B (ORPO preference optimisation on Qwen2.5-7B-Instruct)

This notebook walks through four sections:
1. Dataset overview and task inspection
2. Scoring evaluator — run on three concrete examples
3. Training loss curves
4. Ablation results and bootstrap significance test

**No GPU required for sections 1-2 and 4.** Section 3 is plotting only — no model loading.

---
## Setup

Run from repo root:
```bash
pip install -r requirements.txt
jupyter notebook notebook.ipynb
```

In [ ]:
# ── Colab / local environment setup ─────────────────────────────────────────
# Run this cell first. It installs packages and downloads repo files from
# HuggingFace when running in Google Colab.
import os, sys

IN_COLAB = "google.colab" in sys.modules or "COLAB_GPU" in os.environ
print("Environment:", "Google Colab" if IN_COLAB else "Local")

if IN_COLAB:
    # Install runtime dependencies
    import subprocess
    subprocess.run([
        sys.executable, "-m", "pip", "install", "-q",
        "huggingface_hub", "datasets", "python-dotenv",
        "pydantic", "tqdm", "httpx", "numpy", "pandas",
        "matplotlib", "scipy",
    ], check=True)
    print("Packages installed.")

    from huggingface_hub import hf_hub_download, snapshot_download

    # Download dataset splits from eyorg/tenacious_bench_v0.1
    print("Downloading dataset splits...")
    for split_path in [
        "train/train.jsonl",
        "dev/dev.jsonl",
        "held_out/held_out.jsonl",
    ]:
        local_dir = os.path.dirname(f"tenacious_bench_v0.1/{split_path}")
        os.makedirs(local_dir, exist_ok=True)
        hf_hub_download(
            repo_id="eyorg/tenacious_bench_v0.1",
            repo_type="dataset",
            filename=split_path,
            local_dir="tenacious_bench_v0.1",
        )
    print("  Dataset splits downloaded.")

    # Download code files from eyorg/tenacious-orpo-qwen25-7b model repo
    print("Downloading code files...")
    code_files = [
        "code/scoring_evaluator.py",
        "code/ablations/__init__.py",
        "code/ablations/bootstrap_stats.py",
        "code/ablations/cost_metrics.py",
        "code/ablations/run_ablation.py",
        "code/ablations/ablation_results.json",
        "code/examples/ex1_signal_grounding_pass.json",
        "code/examples/ex2_icp_abstain_correct.json",
        "code/examples/ex3_banned_phrase_fail.json",
        "code/examples/run_examples.py",
        "train_loss.csv",
        "eval_loss.csv",
    ]
    for fname in code_files:
        out_path = fname.replace("code/", "")
        out_dir = os.path.dirname(out_path) if os.path.dirname(out_path) else "."
        os.makedirs(out_dir, exist_ok=True)
        downloaded = hf_hub_download(
            repo_id="eyorg/tenacious-orpo-qwen25-7b",
            repo_type="model",
            filename=fname,
            local_dir=".",
        )
        # Move from code/ prefix to root
        if fname.startswith("code/"):
            dest = fname[len("code/"):]
            os.makedirs(os.path.dirname(dest) if os.path.dirname(dest) else ".", exist_ok=True)
            import shutil
            shutil.move(downloaded, dest)
    # Move CSV files to training/
    os.makedirs("training", exist_ok=True)
    for csv in ["train_loss.csv", "eval_loss.csv"]:
        if os.path.exists(csv):
            shutil.move(csv, f"training/{csv}")
    print("  Code files downloaded.")

    # Set OPENROUTER_API_KEY for section 2 (tone scoring LLM judge)
    # Paste your key here or set via Colab secrets (Runtime > Manage secrets)
    OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "")
    if not OPENROUTER_API_KEY:
        print("  NOTE: OPENROUTER_API_KEY not set — tone dimension will skip LLM judge.")
    os.environ.setdefault("OPENROUTER_API_KEY", OPENROUTER_API_KEY)

print("Setup complete.")


In [1]:
import json
import sys
import csv
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# Ensure repo root is on sys.path
REPO = Path('.').resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

print('Repo root:', REPO)
print('Python:', sys.version)

Repo root: /content
Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


---
## Section 1 — Dataset Overview

In [ ]:
def load_jsonl(path):
    return [json.loads(l) for l in Path(path).read_text(encoding='utf-8').splitlines() if l.strip()]

train = load_jsonl('tenacious_bench_v0.1/train/train.jsonl')
dev   = load_jsonl('tenacious_bench_v0.1/dev/dev.jsonl')
held  = load_jsonl('tenacious_bench_v0.1/held_out/held_out.jsonl')

print(f'Splits: train={len(train)}, dev={len(dev)}, held_out={len(held)}, total={len(train)+len(dev)+len(held)}')

FileNotFoundError: [Errno 2] No such file or directory: 'tenacious_bench_v0.1/train/train.jsonl'

In [ ]:
from collections import Counter

all_tasks = train + dev + held

# Dimension distribution
dims = Counter(t['metadata']['dimension'] for t in all_tasks)
modes = Counter(t['source_mode'] for t in all_tasks)
difficulties = Counter(t['difficulty'] for t in all_tasks)

print('\nDimension counts:')
for d, n in sorted(dims.items(), key=lambda x: -x[1]):
    print(f'  {d:<42} {n}')

print('\nSource mode counts:', dict(modes))
print('Difficulty counts:', dict(difficulties))

In [ ]:
# Visualise dimension distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Dimension bar chart
dim_names = [d.replace('_', '\n') for d in dims.keys()]
axes[0].barh(dim_names, list(dims.values()), color='steelblue', edgecolor='white')
axes[0].set_xlabel('Tasks')
axes[0].set_title('Tasks per Failure Dimension')
axes[0].axvline(22, color='orange', linestyle='--', linewidth=1, label='target=22')
axes[0].legend()

# Source mode pie
axes[1].pie(
    list(modes.values()),
    labels=[m.replace('_', '\n') for m in modes.keys()],
    autopct='%1.0f%%',
    startangle=140,
    colors=['#4c72b0', '#dd8452', '#55a868', '#c44e52']
)
axes[1].set_title('Source Mode Mix (target: 30/30/25/15%)')

plt.tight_layout()
plt.savefig('notebook_fig1_composition.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: notebook_fig1_composition.png')

In [ ]:
# Print one task in full to show the schema
sample = train[0]
print('Sample task fields:', list(sample.keys()))
print('\ntask_id:', sample['task_id'])
print('dimension:', sample['metadata']['dimension'])
print('source_mode:', sample['source_mode'])
print('difficulty:', sample['difficulty'])
print('icp_confidence:', sample['input']['signal_brief']['icp_confidence'])
print('abstain_required:', sample['ground_truth']['abstain_required'])
print('pass_threshold:', sample['scoring_rubric']['pass_threshold'])
print('\nIdeal output (first 200 chars):',
      sample['ground_truth']['ideal_output'][:200])

---
## Section 2 — Scoring Evaluator

Run `score_task()` on the three hand-authored example tasks.
These are the same tasks described in the interim report and `.claude/Eyor_interim_report.md`.

**Expected results (deterministic for lexical dimensions; LLM judge adds slight variance for tone):**
- `TB-EX-001` (trace_derived, signal grounding): PASS ~0.87
- `TB-EX-002` (programmatic, ICP abstain): PASS ~1.00  
- `TB-EX-003` (adversarial P03, banned phrases): FAIL ~0.31

In [ ]:
from scoring_evaluator import score_task

examples = [
    json.loads(Path('examples/ex1_signal_grounding_pass.json').read_text()),
    json.loads(Path('examples/ex2_icp_abstain_correct.json').read_text()),
    json.loads(Path('examples/ex3_banned_phrase_fail.json').read_text()),
]

results = []
for task in examples:
    r = score_task(task)
    results.append(r)
    status = 'PASS' if r.passed else 'FAIL'
    print(f'{r.task_id}  {status}  total={r.total_score:.3f}  threshold={r.pass_threshold}  latency={r.latency_ms:.0f}ms  model_calls={r.model_calls}')

In [ ]:
# Per-dimension breakdown for all three examples
for r, task in zip(results, examples):
    print(f'\n=== {r.task_id} (source_mode={task["source_mode"]}, dim={task["metadata"]["dimension"]}) ===')
    print(f'    Total: {r.total_score:.3f}  Passed: {r.passed}')
    for d in r.dimensions:
        bar = '#' * int(d.raw_score * 20)
        print(f'  {d.name:<30} raw={d.raw_score:.2f}  w={d.weight:.2f}  weighted={d.weighted:.3f}')
        print(f'    [{bar:<20}]  {d.reason[:80]}')

In [ ]:
# Bar chart: per-dimension scores for all three examples
fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
titles = [
    'TB-EX-001 (trace_derived, PASS)',
    'TB-EX-002 (programmatic, PASS)',
    'TB-EX-003 (adversarial, FAIL)',
]
colors = ['#2ecc71', '#2ecc71', '#e74c3c']

for ax, r, title, color in zip(axes, results, titles, colors):
    dim_names = [d.name.replace('_', '\n') for d in r.dimensions]
    raw_scores = [d.raw_score for d in r.dimensions]
    bars = ax.bar(dim_names, raw_scores, color=color, alpha=0.8, edgecolor='white')
    ax.axhline(r.pass_threshold, color='navy', linestyle='--', linewidth=1.2,
               label=f'threshold={r.pass_threshold}')
    ax.set_ylim(0, 1.15)
    ax.set_title(f'{title}\ntotal={r.total_score:.3f}', fontsize=10)
    ax.set_ylabel('Raw score' if ax == axes[0] else '')
    ax.tick_params(axis='x', labelsize=7)
    ax.legend(fontsize=8)

plt.suptitle('TenaciousBench: Per-Dimension Scores for Three Example Tasks', fontsize=12)
plt.tight_layout()
plt.savefig('notebook_fig2_example_scores.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: notebook_fig2_example_scores.png')

---
## Section 3 — Training Loss Curves

Plots training and eval loss from the ORPO training run logs.
These are read from `training/train_loss.csv` and `training/eval_loss.csv`.

In [ ]:
train_loss_df = pd.read_csv('training/train_loss.csv')
eval_loss_df  = pd.read_csv('training/eval_loss.csv')

print('Train loss rows:', len(train_loss_df))
print(train_loss_df.head())
print('\nEval loss rows:', len(eval_loss_df))
print(eval_loss_df)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(train_loss_df['step'], train_loss_df['loss'],
        color='steelblue', linewidth=2, label='Train loss')
ax.scatter(eval_loss_df['step'], eval_loss_df['eval_loss'],
           color='darkorange', s=80, zorder=5, label='Eval loss (checkpoint)')
ax.plot(eval_loss_df['step'], eval_loss_df['eval_loss'],
        color='darkorange', linewidth=1.5, linestyle='--')

# Mark kill criterion threshold
ax.axhline(0.35, color='red', linestyle=':', linewidth=1.2, label='Kill criterion (eval_loss=0.35)')
ax.annotate('Kill criterion\ncheck at step 100',
            xy=(100, 0.35), xytext=(60, 0.40),
            fontsize=8, color='red',
            arrowprops=dict(arrowstyle='->', color='red', lw=1))

# Mark epoch boundaries
for epoch_step, epoch_label in [(37, 'Epoch 1'), (74, 'Epoch 2'), (110, 'Epoch 3')]:
    ax.axvline(epoch_step, color='gray', linestyle=':', linewidth=0.8, alpha=0.7)
    ax.text(epoch_step + 1, 0.56, epoch_label, fontsize=8, color='gray')

ax.set_xlabel('Training step')
ax.set_ylabel('Loss')
ax.set_title('ORPO Training — Qwen2.5-7B-Instruct + LoRA r=16\n'
             '110 preference pairs, 3 epochs, beta=0.1, lr=8e-6 cosine')
ax.legend()
ax.set_ylim(0.20, 0.60)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
plt.tight_layout()
plt.savefig('notebook_fig3_training_loss.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: notebook_fig3_training_loss.png')

---
## Section 4 — Ablation Results and Bootstrap Significance

Reads pre-computed results from `ablations/ablation_results.json` and runs
the paired bootstrap test to verify significance.

In [ ]:
with open('ablations/ablation_results.json') as f:
    abl = json.load(f)

print('Benchmark:', abl['benchmark'])
print('Held-out tasks:', abl['num_held_out_tasks'])
print('Bootstrap resamples:', abl['bootstrap_resamples'])
print()

print('Baseline pass@1:', abl['baseline']['held_out_pass_at_1'])
print('Delta A pass@1:', abl['delta_a']['held_out_pass_at_1'],
      f"(+{abl['delta_a']['lift_pp']} pp, p={abl['delta_a']['p_value']})")
print('Delta B pass@1:', abl['delta_b']['held_out_pass_at_1'],
      f"(+{abl['delta_b']['lift_pp']} pp, p={abl['delta_b']['p_value']})")

In [ ]:
# Run the bootstrap test from bootstrap_stats.py
from ablations.bootstrap_stats import compute_all_from_results_file
compute_all_from_results_file('ablations/ablation_results.json', n_resamples=10000)

In [ ]:
# Per-dimension comparison: Delta A vs Baseline
dims_ordered = list(abl['delta_a']['per_dimension'].keys())
delta_a_scores = [abl['delta_a']['per_dimension'][d] for d in dims_ordered]
# Baseline per-dimension not stored; use uniform baseline pass rate as reference
baseline_rate = abl['baseline']['held_out_pass_at_1']

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(dims_ordered))
width = 0.35

ax.bar(x, delta_a_scores, width, label='Delta A (ORPO)', color='#2ecc71', alpha=0.85)
ax.axhline(baseline_rate, color='steelblue', linestyle='--', linewidth=1.5,
           label=f'Baseline pass@1={baseline_rate:.2f}')
ax.axhline(0.70, color='gray', linestyle=':', linewidth=1,
           label='pass_threshold=0.70')

ax.set_xticks(x)
ax.set_xticklabels([d.replace('_', '\n') for d in dims_ordered], fontsize=7)
ax.set_ylabel('pass@1')
ax.set_ylim(0.75, 1.0)
ax.set_title('Delta A (ORPO) vs Baseline — Per-Dimension pass@1\nHeld-out split, n=44 tasks')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('notebook_fig4_ablation_perdim.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: notebook_fig4_ablation_perdim.png')

In [ ]:
# Cost-quality Pareto chart
from ablations.cost_metrics import compute_pareto
pareto_rows = compute_pareto(abl)

labels = [r['variant'] for r in pareto_rows]
pass_rates = [r['pass_at_1'] for r in pareto_rows]
costs = [r['cost_usd_per_task'] * 1000 for r in pareto_rows]  # milliUSD

fig, ax = plt.subplots(figsize=(8, 5))
scatter_colors = ['#2ecc71', '#f39c12', '#e74c3c']
for i, (label, pr, cost, color) in enumerate(zip(labels, pass_rates, costs, scatter_colors)):
    ax.scatter(cost, pr, s=160, color=color, zorder=5, label=label)
    ax.annotate(f'  {label}\n  pass@1={pr:.3f}', (cost, pr), fontsize=8)

ax.set_xlabel('Cost per task (milli-USD)')
ax.set_ylabel('pass@1')
ax.set_title('Cost-Quality Pareto — TenaciousBench Held-Out\nLower-left = better cost; higher = better quality')
ax.legend()
plt.tight_layout()
plt.savefig('notebook_fig5_cost_pareto.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: notebook_fig5_cost_pareto.png')

---
## Summary

| Figure | Content |
|--------|---------|
| `notebook_fig1_composition.png` | Dataset dimension distribution + source mode pie |
| `notebook_fig2_example_scores.png` | Per-dimension scores for 3 example tasks |
| `notebook_fig3_training_loss.png` | ORPO training and eval loss curves |
| `notebook_fig4_ablation_perdim.png` | Delta A vs baseline per dimension |
| `notebook_fig5_cost_pareto.png` | Cost-quality Pareto scatter |

**All claims in this notebook are backed by `evidence_graph.json`.**